In [10]:
"""
<td>
    <div class="obc-tmpl-character__voice-item obc-tmpl-character__play-voice">
        <div class="obc-tmpl-character__voice-btn">
        </div> 
        <span class="obc-tmpl-character__voice-content"> 	
            [text] 	 
        </span>
    </div>
 </td>
"""

'\n<td>\n    <div class="obc-tmpl-character__voice-item obc-tmpl-character__play-voice">\n        <div class="obc-tmpl-character__voice-btn">\n        </div> \n        <span class="obc-tmpl-character__voice-content"> \t\n            [text] \t \n        </span>\n    </div>\n </td>\n'

In [11]:
import time
from seleniumwire import webdriver  # pip install selenium-wire
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver import ActionChains

# Configuration constants
GECKODRIVER_PATH = "/home/co/firefox/geckodriver"
FIREFOX_BINARY = "/home/co/firefox/firefox/firefox"
# columbina
URL = "https://baike.mihoyo.com/ys/obc/content/507505/detail"
# layla
URL = "https://baike.mihoyo.com/ys/obc/content/5297/detail"
OUTPUT_DIR = "/dev/shm/genshin_audio"

# Define a request interceptor to add "Cache-Control: no-cache" to every request.
def interceptor(request):
    request.headers['Cache-Control'] = 'no-cache'

options = Options()
options.binary_location = "/home/co/firefox/firefox/firefox"
options.add_argument('--headless')
options.add_argument("--width=1920")
options.add_argument("--height=1080")

# Initialize selenium-wire Firefox driver with our interceptor.
driver = webdriver.Firefox(
    service=Service("/home/co/firefox/geckodriver"),
    options=options,
)
driver.request_interceptor = interceptor

In [ ]:
# Navigate to the target URL and wait until at least one container is present.
driver.get(URL)
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "div.obc-tmpl-character__voice-item"))
)

# Locate all container elements that include both the text and the voice button.
containers = driver.find_elements(
    By.CSS_SELECTOR,
    "div.obc-tmpl-character__voice-item.obc-tmpl-character__play-voice"
)

results = []
failed=0

for index, container in enumerate(containers):
    # Extract text from the container.
    try:
        text_el = container.find_element(By.CSS_SELECTOR, "span.obc-tmpl-character__voice-content")
        text = text_el.text.strip()
    except Exception:
        text = ""
    
    # Locate the voice button.
    try:
        button = container.find_element(By.CSS_SELECTOR, "div.obc-tmpl-character__voice-btn")
    except Exception:
        print(f"Button not found in container {index}")
        results.append((text, None))
        continue

    # Scroll the container into view.
    driver.execute_script("arguments[0].scrollIntoView(true);", container)
    
    mp3_url = ""
    max_attempts = 10  # Total wait: ~5 seconds (10*0.5)
    attempt = 0

    # Clear previous network requests.
    del driver.requests

    while attempt < max_attempts and not mp3_url:
        try:
            ActionChains(driver).click(button).perform()
        except Exception as e:
            print(f"Error clicking button in container {index}: {e}")
        # Wait a bit for network requests to occur.
        time.sleep(0.5)
        # Check driver.requests for a GET request with URL ending in .mp3.
        for request in driver.requests:
            if request.method == "GET" and request.url.lower().endswith(".mp3"):
                mp3_url = request.url
                print(f"Captured mp3 for container {index}: {mp3_url}")
                break
        attempt += 1
        driver.requests.clear()

    if not mp3_url:
        print(f"No mp3 request captured for container {index}")
        failed+=1
    results.append((text, mp3_url))

driver.quit()
print(f"# of FAILED: {failed}")

Captured mp3 for container 0: https://uploadstatic.mihoyo.com/ys-obc/2022/11/18/16576950/d70b91ada27cbb1dfdb5dae6413045be_1059314131526931846.mp3
Captured mp3 for container 1: https://uploadstatic.mihoyo.com/ys-obc/2022/11/18/16576950/6454d3ed347aa62ec368bafefdca95ac_3943454212375297890.mp3
Captured mp3 for container 2: https://uploadstatic.mihoyo.com/ys-obc/2022/11/18/16576950/de15cf150679f0b24377bad2009c3bd2_3944832558537452691.mp3
Captured mp3 for container 3: https://uploadstatic.mihoyo.com/ys-obc/2022/11/18/16576950/df7ee873899e9a855a572e2921522da9_50131164358505477.mp3
Captured mp3 for container 4: https://uploadstatic.mihoyo.com/ys-obc/2022/11/18/16576950/68fece013e067483ea15a2ca1b0edcd3_4790914243404175672.mp3
Captured mp3 for container 5: https://uploadstatic.mihoyo.com/ys-obc/2022/11/18/16576950/002219c97380fd3d4308b10e4fbccdeb_2236920489457451632.mp3
Captured mp3 for container 6: https://uploadstatic.mihoyo.com/ys-obc/2022/11/18/16576950/d41011d65509d5f9255396a22cc3399c_7102

In [ ]:
len(list(set([y for x,y in results])))==len(results)

True

In [ ]:
import os
import requests
quarter = len(results) // 4
marks=["C","J","E","K"]
# Headers from network inspection
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:136.0) Gecko/20100101 Firefox/136.0",
    "Referer": "https://bbs.mihoyo.com/",
    "Accept": "audio/webm,audio/ogg,audio/wav,audio/*;q=0.9,application/ogg;q=0.7,video/*;q=0.6,*/*;q=0.5",
    "DNT": "1",
    "Sec-Fetch-Dest": "audio",
    "Sec-Fetch-Mode": "no-cors",
    "Sec-Fetch-Site": "same-site"
}
person_id=URL.split('/')[-2]
len(results),quarter,person_id

(304, 76, '507505')

In [ ]:
import os
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()

# Retry strategy (handles DNS fail, resets, 5xx, etc.)
retries = Retry(
    total=5,
    backoff_factor=1.5,
    status_forcelist=[403, 429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retries)
session.mount("http://", adapter)
session.mount("https://", adapter)

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://wiki.hoyolab.com/",  # CDN sometimes checks this
    "Accept": "*/*",
    "Connection": "keep-alive",
}

TIMEOUT = (5, 30)  # connect timeout, read timeout


for i, (text, url) in enumerate(results):
    if not url:
        continue

    q = i // quarter
    dir_path = os.path.join(OUTPUT_DIR, person_id, marks[q])
    os.makedirs(dir_path, exist_ok=True)

    out_path = os.path.join(dir_path, f"{i - q*quarter}.mp3")
    tmp_path = out_path + ".part"

    try:
        with session.get(url, headers=HEADERS, stream=True, timeout=TIMEOUT) as r:
            r.raise_for_status()

            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

        os.replace(tmp_path, out_path)  # atomic rename
        print(f"✓ Downloaded {i}")

    except Exception as e:
        print(f"✗ Failed {i}: {e}")
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

        #time.sleep(1)  # small delay to avoid CDN rate limits

✓ Downloaded 0
✓ Downloaded 1
✓ Downloaded 2
✓ Downloaded 3
✓ Downloaded 4
✓ Downloaded 5
✓ Downloaded 6
✓ Downloaded 7
✓ Downloaded 8
✓ Downloaded 9
✓ Downloaded 10
✓ Downloaded 11
✓ Downloaded 12
✓ Downloaded 13
✓ Downloaded 14
✓ Downloaded 15
✓ Downloaded 16
✓ Downloaded 17
✓ Downloaded 18
✓ Downloaded 19
✓ Downloaded 20
✓ Downloaded 21
✓ Downloaded 22
✓ Downloaded 23
✓ Downloaded 24
✓ Downloaded 25
✓ Downloaded 26
✓ Downloaded 27
✓ Downloaded 28
✓ Downloaded 29
✓ Downloaded 30
✓ Downloaded 31
✓ Downloaded 32
✓ Downloaded 33
✓ Downloaded 34
✓ Downloaded 35
✓ Downloaded 36
✓ Downloaded 37
✓ Downloaded 38
✓ Downloaded 39
✓ Downloaded 40
✓ Downloaded 41
✓ Downloaded 42
✓ Downloaded 43
✓ Downloaded 44
✓ Downloaded 45
✓ Downloaded 46
✓ Downloaded 47
✓ Downloaded 48
✓ Downloaded 49
✓ Downloaded 50
✓ Downloaded 51
✓ Downloaded 52
✓ Downloaded 53
✓ Downloaded 54
✓ Downloaded 55
✓ Downloaded 56
✓ Downloaded 57
✓ Downloaded 58
✓ Downloaded 59
✓ Downloaded 60
✓ Downloaded 61
✓ Downloaded 62
✓ 

In [ ]:
dir=os.path.join(OUTPUT_DIR,person_id)
os.makedirs(dir, exist_ok=True)
with open(os.path.join(dir,"index.csv"),"w") as f:
    i=0
    while i<quarter:
        f.write(f"{i},{results[i][0]}\n")
        i+=1
